# تمرین ۱ — دیتاست دوم: DeepSentiPers (۵ کلاس)

ParsBERT در برابر XLM-RoBERTa، با همون تنظیمات دیتاست اول (اسنپ‌فود)، برای مقایسه‌ی مستقیم.

دیتاست: [Khedesh/DeepSentiPers](https://huggingface.co/datasets/Khedesh/DeepSentiPers) — نظرات فارسی محصولات دیجیتال، ۵ کلاس (furious/angry/neutral/happy/delighted).


## ۰ — GPU


In [ ]:
# بررسی اینکه آیا GPU در دسترس است یا نه
!nvidia-smi

## ۱ — نصب کتابخونه‌ها


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## ۲ — Import


In [ ]:
import numpy as np
import pandas as pd
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

print("همه‌ی کتابخونه‌ها با موفقیت import شدند.")

## ۳ — بارگذاری دیتاست

برچسب‌های اصلی از `-2` تا `2` هستن (furious=-2 … delighted=2).


In [ ]:
DATASET_NAME = "Khedesh/DeepSentiPers"  # نسخه‌ای که واقعاً ۵ کلاس کامل داره

dataset = load_dataset(DATASET_NAME)
print(dataset)


In [ ]:
# اسم دقیق ستون‌ها را ببینیم
split_name = list(dataset.keys())[0]   # اولین split موجود (معمولاً "train")
print("ستون‌های موجود:", dataset[split_name].column_names)
print("\nیک نمونه از داده:")
print(dataset[split_name][0])

In [ ]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

from datasets import Value

available_columns = dataset[split_name].column_names
text_candidates = ["text", "review", "comment", "sentence", "body", "content"]
label_candidates = ["label", "labels", "sentiment", "label_id", "emotion", "class"]

if TEXT_COLUMN not in available_columns:
    for c in text_candidates:
        if c in available_columns:
            TEXT_COLUMN = c; break
if LABEL_COLUMN not in available_columns:
    for c in label_candidates:
        if c in available_columns:
            LABEL_COLUMN = c; break

label_feature = dataset[split_name].features[LABEL_COLUMN]
if not str(getattr(label_feature, "dtype", "")).startswith(("int", "float")):
    dataset = dataset.class_encode_column(LABEL_COLUMN)
dataset = dataset.cast_column(LABEL_COLUMN, Value("int64"))

# شیفت به 0..n-1 (چون برچسب‌ها منفی هم دارن)
all_values = []
for split in dataset.keys():
    all_values.extend(dataset[split][LABEL_COLUMN])
min_label = min(all_values)
if min_label != 0:
    dataset = dataset.map(lambda ex: {LABEL_COLUMN: ex[LABEL_COLUMN] - min_label})

if LABEL_COLUMN != "labels":
    dataset = dataset.rename_column(LABEL_COLUMN, "labels")
    LABEL_COLUMN = "labels"

all_labels = set()
for split in dataset.keys():
    all_labels.update(dataset[split][LABEL_COLUMN])
num_labels = len(all_labels)
print("تعداد کلاس‌ها:", num_labels, "| مقادیر:", sorted(all_labels))


## ۴ — تقسیم train / test


In [ ]:
if "test" in dataset:
    train_data = dataset["train"]
    test_data = dataset["test"]
elif "validation" in dataset:
    train_data = dataset["train"]
    test_data = dataset["validation"]
else:
    split_dataset = dataset[split_name].train_test_split(test_size=0.2, seed=42)
    train_data, test_data = split_dataset["train"], split_dataset["test"]

print("train:", len(train_data), "| test:", len(test_data))


## ۵ — Tokenization


In [ ]:
MODEL_NAME_1 = "HooshvareLab/bert-fa-base-uncased"   # ParsBERT — مدل اول

tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_NAME_1)

def make_tokenize_fn(tokenizer):
    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_COLUMN],
            padding="max_length",
            truncation=True,
            max_length=128,
        )
    return tokenize_fn

tokenize_fn_1 = make_tokenize_fn(tokenizer_1)

train_tokenized_1 = train_data.map(tokenize_fn_1, batched=True)
test_tokenized_1 = test_data.map(tokenize_fn_1, batched=True)

print(train_tokenized_1[0].keys())

In [ ]:
model_1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_1,
    num_labels=num_labels,
    problem_type="single_label_classification",  # صراحتاً می‌گیم طبقه‌بندی تک‌برچسبیه (نه رگرسیون/multi-label)
)

## ۷ — معیار ارزیابی


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

## ۸ — TrainingArguments و Trainer


In [ ]:
training_args_1 = TrainingArguments(
    output_dir="./results_model1",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none",   # جلوگیری از اتصال خودکار به سرویس‌های لاگ بیرونی
)

trainer_1 = Trainer(
    model=model_1,
    args=training_args_1,
    train_dataset=train_tokenized_1,
    eval_dataset=test_tokenized_1,
    compute_metrics=compute_metrics,
)

## ۹ — آموزش مدل اول (ParsBERT)


In [ ]:
trainer_1.train()

## ۱۰ — ارزیابی مدل اول


In [ ]:
import time

start_time = time.time()
results_model1 = trainer_1.evaluate()
elapsed_model1 = time.time() - start_time

print(results_model1)
print(f"\nزمان تقریبی این اجرا (ارزیابی): {elapsed_model1:.1f} ثانیه")
print("زمان کامل آموزش را از لاگ‌های بالای سلول trainer_1.train() (ستون Runtime) هم می‌توانی ببینی.")

## ۱۱ — مدل دوم (XLM-RoBERTa)


In [ ]:
MODEL_NAME_2 = "xlm-roberta-base"   # XLM-RoBERTa — مدل دوم

# ۵) توکنایز کردن داده با tokenizer مدل دوم
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_NAME_2)
tokenize_fn_2 = make_tokenize_fn(tokenizer_2)

train_tokenized_2 = train_data.map(tokenize_fn_2, batched=True)
test_tokenized_2 = test_data.map(tokenize_fn_2, batched=True)

# ۶) بارگذاری مدل دوم برای طبقه‌بندی
model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_2,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

# ۸) همان تنظیمات آموزشی، فقط output_dir را جدا می‌کنیم که با مدل اول قاطی نشود
training_args_2 = TrainingArguments(
    output_dir="./results_model2",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none",
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_tokenized_2,
    eval_dataset=test_tokenized_2,
    compute_metrics=compute_metrics,
)

# ۹) آموزش مدل دوم
trainer_2.train()

In [ ]:
# ۱۰) ارزیابی نهایی مدل دوم
start_time = time.time()
results_model2 = trainer_2.evaluate()
elapsed_model2 = time.time() - start_time

print(results_model2)
print(f"\nزمان تقریبی این اجرا (ارزیابی): {elapsed_model2:.1f} ثانیه")

## ۱۲ — جدول مقایسه


In [ ]:
comparison_table = pd.DataFrame([
    {
        "دیتاست": DATASET_NAME,
        "مدل": "ParsBERT",
        "Accuracy": round(results_model1["eval_accuracy"], 4),
        "F1-score": round(results_model1["eval_f1"], 4),
    },
    {
        "دیتاست": DATASET_NAME,
        "مدل": "XLM-RoBERTa",
        "Accuracy": round(results_model2["eval_accuracy"], 4),
        "F1-score": round(results_model2["eval_f1"], 4),
    },
])

comparison_table

**نتایج نهایی (DeepSentiPers):**

| مدل | Accuracy | F1-score |
|---|---|---|
| ParsBERT | 0.7325 | 0.7344 |
| XLM-RoBERTa | 0.7395 | 0.7388 |


## ۱۳ — پاسخ سوالات تحلیلی (هر دو دیتاست)

**۱. کدام مدل بهتر بود؟**
روی هر دو دیتاست، XLM-RoBERTa کمی از ParsBERT بهتر بود (اسنپ‌فود: 0.8776 در برابر 0.8680؛ DeepSentiPers: 0.7395 در برابر 0.7325). دلیل: پیش‌آموزش بزرگ‌تر و چندزبانه‌ی XLM-RoBERTa باعث نمایش‌های زبانی پایدارتری شده، حتی برای یک تکلیف تک‌زبانه.

**۲. نتیجه روی دو دیتاست یکسان بود یا فرق داشت؟**
جهت مقایسه (XLM-RoBERTa > ParsBERT) در هر دو دیتاست یکسان بود، اما مقدار مطلق دقت روی DeepSentiPers (~۷۳٪) خیلی پایین‌تر از اسنپ‌فود (~۸۷٪) بود — چون DeepSentiPers پنج‌کلاسه و مرزهای تصمیم ظریف‌تری داره (مثلاً angry در برابر furious)، و حجم داده‌ی آموزشش هم کمتره (~۷هزار در برابر ده‌ها هزار نمونه‌ی اسنپ‌فود). فاصله‌ی بین دو مدل هم روی DeepSentiPers کمتر بود (۰.۷ واحد درصد در برابر ~۱ واحد درصد در اسنپ‌فود).

**۳. مزایا/معایب تک‌زبانه در برابر چندزبانه؟**
ParsBERT روی اسنپ‌فود بین epoch ۲ و ۳ overfit کرد؛ روی DeepSentiPers این overfitting خفیف‌تر بود ولی بازم validation loss در epoch ۳ کمی بالا رفت (۰.۷۰۹ → ۰.۷۳۱). XLM-RoBERTa روی هر دو دیتاست منحنی یادگیری صاف‌تر و بدون overfitting داشت — نشون می‌ده تنوع بیشتر داده‌ی پیش‌آموزشش اثر تنظیم‌کننده (regularization) هم داره.

**۴. اثر فرضی تغییر learning rate؟**
نرخ یادگیری بین دو مدل و دو دیتاست ثابت نگه داشته شد (2e-5) تا مقایسه منصفانه باشه. با توجه به رفتار ParsBERT (بخصوص روی اسنپ‌فود)، نرخ بزرگ‌تر احتمالاً بی‌ثباتی بیشتری ایجاد می‌کرد؛ نرخ کوچیک‌تر با ۳ epoch محدود، وقت کافی برای یادگیری نمی‌داد. این فرض به‌صورت عملی تست نشده است.
